# U-Net++（PASCAL VOC 2007を用いたセマンティックセグメンテーション）

---
## 目的
U-Netを拡張したU-Net++を，PASCAL VOC 2007データセットを用いて実装する．U-Netのエンコーダ・デコーダ間に，ネストされた密なskip connection（グリッド状に配置された中間ノード $X^{i,j}$）を導入することで，エンコーダとデコーダの特徴の意味的なギャップを段階的に埋める仕組みを理解する．

## モジュールのインポート
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
!pip install -q torchmetrics

import os
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.datasets import VOCSegmentation
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torchmetrics.classification import MulticlassJaccardIndex
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込みます．20クラスの物体に対して画素単位のクラスラベルが付与されており，学習用（trainval）422枚，評価用（test）210枚の画像から構成されます．データセットの読み込みやDataLoaderの基本的な使い方については`02_dnn_simple_pytorch/existing_dataset.ipynb`を，VOCデータセットの詳細については`fcn.ipynb`を参照してください．

Googleドライブに配置したVOCdevkit（2007）のzipファイルを`gdown`でダウンロードし，`torchvision.datasets.VOCSegmentation`で読み込みます．

In [ ]:
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
n_class = len(VOC_CLASSES)  # 21（背景を含む）
IGNORE_INDEX = 255  # 物体境界などの無視領域
IMG_SIZE = 256

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


class VOCSegmentationDataset(torch.utils.data.Dataset):
    """VOCSegmentationをラップし，画像とマスクを同じサイズにリサイズしたTensorとして返すDataset"""
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCSegmentation(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, mask = self.voc[idx]
        image = self.img_transform(image)
        # マスクは値がそのままクラスIDのため，補間で値が混ざらないよう最近傍補間でリサイズする
        mask = TF.resize(mask, [self.img_size, self.img_size], interpolation=TF.InterpolationMode.NEAREST)
        mask = torch.from_numpy(np.array(mask, dtype=np.int64))
        return image, mask


batch_size = 8
train_data = VOCSegmentationDataset('trainval')
test_data = VOCSegmentationDataset('test')
print('train:', len(train_data), 'test:', len(test_data))

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

image, mask = train_data[0]
print('image:', image.shape, 'mask:', mask.shape, 'mask unique:', torch.unique(mask))

## VOCカラーパレットと可視化用関数
クラスIDの2次元配列（マスク）をそのまま画像として表示しても分かりにくいため，各クラスIDに固有の色を割り当ててカラー画像に変換する関数を用意します．ここでは，VOCdevkitで標準的に使われているビット演算によるカラーパレットの生成方法を使用します．

In [ ]:
def voc_colormap(n=n_class):
    """VOCdevkitで標準的に使われる、クラスIDから固有のRGB色を生成する関数"""
    cmap = np.zeros((n, 3), dtype=np.uint8)
    for i in range(n):
        r = g = b = 0
        c = i
        for j in range(8):
            r |= ((c >> 0) & 1) << (7 - j)
            g |= ((c >> 1) & 1) << (7 - j)
            b |= ((c >> 2) & 1) << (7 - j)
            c >>= 3
        cmap[i] = [r, g, b]
    return cmap


VOC_COLORMAP = voc_colormap()


def decode_segmap(mask):
    """クラスIDのマスク(H, W)を，可視化用のカラー画像(H, W, 3)に変換する"""
    mask = mask.clone()
    mask[mask == IGNORE_INDEX] = 0  # 可視化のため，無視領域は背景色として扱う
    return VOC_COLORMAP[mask.cpu().numpy()]

## U-Net++
U-Netは，エンコーダの各解像度の特徴マップを，デコーダの対応する解像度に1本のskip connectionでconcatする構造でした．しかし，エンコーダの浅い層の特徴（低レベルで解像度が高い）と，デコーダ側の特徴（高レベルの意味情報を含むがアップサンプリングされたもの）は意味的なギャップが大きく，それらを単純にconcatするだけでは十分に特徴を融合できないという問題があります．

U-Net++は，エンコーダとデコーダの間に**ネストされた密なskip connection**を挿入することで，このギャップを段階的に埋めます．具体的には，各解像度レベル $i$（$i=0$が最も高解像度，$i=4$が最も低解像度）と，畳み込みブロックの段数 $j$ で添字付けられた中間ノード $X^{i,j}$ をグリッド状に配置します．

- $X^{i,0}$（$j=0$）：通常のU-Netと同じエンコーダのダウンサンプリング経路の出力．
- $X^{i,j}$（$j \geq 1$）：同じ解像度レベル $i$ の既存ノード $X^{i,0}, \dots, X^{i,j-1}$ **すべて**と，1段深いレベル $i+1$ の $X^{i+1,j-1}$ をアップサンプリングしたものをconcatし，畳み込みブロックを適用したもの．
- 最終的な出力は，最も高解像度なレベル（$i=0$）かつ最終列の $X^{0,4}$ から得ます．

U-Netとの違いは，**デコーダの各ノードが1本のskipではなく，同じ解像度上に蓄積された複数の中間特徴を経由して段階的に特徴を融合する点**です。U-Netでは $X^{i+1,j-1}$ をアップサンプリングしてエンコーダの $X^{i,0}$ 1本とだけconcatしますが，U-Net++では $X^{i,0}$ から $X^{i,j-1}$ までの**すべての中間ノード**をconcatするため，同じ解像度レベル内で特徴が密に再利用されます．

原論文（Zhou et al., 2018）では，各列 $X^{0,1}, X^{0,2}, X^{0,3}, X^{0,4}$ の出力すべてに損失をかけるDeep Supervisionという学習方法が提案されていますが，本ノートブックでは実装を簡略化し，最終列の $X^{0,4}$ の出力のみを用いた通常の学習を行います．

### 畳み込みブロック
グリッド上のすべてのノード $X^{i,j}$ で共通して使用する，3x3畳み込み×2から成る基本ブロックを定義します（U-Netと同様の構成）．

In [ ]:
class ConvBlock(nn.Module):
    """3x3畳み込み + BatchNorm + ReLU を2回繰り返す，U-Net++のグリッド上の全ノード共通の基本ブロック"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

### ネストされたグリッド構造の実装
深さ4（解像度レベル $i=0,\dots,4$，チャンネル数は64, 128, 256, 512, 1024）のグリッドをスクラッチで実装します．各ノード $X^{i,j}$ を個別の`ConvBlock`として保持し，`forward`内でグリッドを列（$j=0 \to 4$）ごとに計算していきます．ノード $X^{i,j}$（$j \geq 1$）の入力チャンネル数は，「同じレベル $i$ の既存ノード $j$ 個分（各`ch[i]`）」+「1段深いレベルからアップサンプリングした特徴（`ch[i+1]`）」になる点に注意してください．

In [ ]:
class UNetPlusPlus(nn.Module):
    """U-Net++（Deep Supervisionなし，X^{0,4}の出力のみを使用）"""
    def __init__(self, n_class=n_class, in_ch=3):
        super().__init__()
        ch = [64, 128, 256, 512, 1024]
        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        # 列 j=0：通常のU-Netと同じエンコーダのダウンサンプリング経路
        self.conv0_0 = ConvBlock(in_ch, ch[0])
        self.conv1_0 = ConvBlock(ch[0], ch[1])
        self.conv2_0 = ConvBlock(ch[1], ch[2])
        self.conv3_0 = ConvBlock(ch[2], ch[3])
        self.conv4_0 = ConvBlock(ch[3], ch[4])

        # 列 j=1：各レベルでX^{i,0}とアップサンプリングしたX^{i+1,0}をconcat
        self.conv0_1 = ConvBlock(ch[0] * 1 + ch[1], ch[0])
        self.conv1_1 = ConvBlock(ch[1] * 1 + ch[2], ch[1])
        self.conv2_1 = ConvBlock(ch[2] * 1 + ch[3], ch[2])
        self.conv3_1 = ConvBlock(ch[3] * 1 + ch[4], ch[3])

        # 列 j=2：同じレベルの既存ノード2個 + アップサンプリングした1段深いノード
        self.conv0_2 = ConvBlock(ch[0] * 2 + ch[1], ch[0])
        self.conv1_2 = ConvBlock(ch[1] * 2 + ch[2], ch[1])
        self.conv2_2 = ConvBlock(ch[2] * 2 + ch[3], ch[2])

        # 列 j=3
        self.conv0_3 = ConvBlock(ch[0] * 3 + ch[1], ch[0])
        self.conv1_3 = ConvBlock(ch[1] * 3 + ch[2], ch[1])

        # 列 j=4（最終列）
        self.conv0_4 = ConvBlock(ch[0] * 4 + ch[1], ch[0])

        self.final = nn.Conv2d(ch[0], n_class, kernel_size=1)

    def forward(self, x):
        x0_0 = self.conv0_0(x)
        x1_0 = self.conv1_0(self.pool(x0_0))
        x0_1 = self.conv0_1(torch.cat([x0_0, self.up(x1_0)], dim=1))

        x2_0 = self.conv2_0(self.pool(x1_0))
        x1_1 = self.conv1_1(torch.cat([x1_0, self.up(x2_0)], dim=1))
        x0_2 = self.conv0_2(torch.cat([x0_0, x0_1, self.up(x1_1)], dim=1))

        x3_0 = self.conv3_0(self.pool(x2_0))
        x2_1 = self.conv2_1(torch.cat([x2_0, self.up(x3_0)], dim=1))
        x1_2 = self.conv1_2(torch.cat([x1_0, x1_1, self.up(x2_1)], dim=1))
        x0_3 = self.conv0_3(torch.cat([x0_0, x0_1, x0_2, self.up(x1_2)], dim=1))

        x4_0 = self.conv4_0(self.pool(x3_0))
        x3_1 = self.conv3_1(torch.cat([x3_0, self.up(x4_0)], dim=1))
        x2_2 = self.conv2_2(torch.cat([x2_0, x2_1, self.up(x3_1)], dim=1))
        x1_3 = self.conv1_3(torch.cat([x1_0, x1_1, x1_2, self.up(x2_2)], dim=1))
        x0_4 = self.conv0_4(torch.cat([x0_0, x0_1, x0_2, x0_3, self.up(x1_3)], dim=1))

        return self.final(x0_4)

## 損失関数
セグメンテーションは，画素ごとのクラス分類問題とみなせるため，`nn.CrossEntropyLoss`をそのまま利用できます．マスクに含まれる無視領域（ラベル255）は`ignore_index`で損失計算から除外します．

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

## ネットワークの作成
U-Net++を構築し，最適化手法としてAdamを設定します．事前学習済みバックボーンを使わずスクラッチで学習するため，事前学習済みVGG16を使うFCNなどより大きめの学習率（`1e-3`）を用います．

In [ ]:
model = UNetPlusPlus(n_class=n_class).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

num_params = sum(p.numel() for p in model.parameters())
print('パラメータ数:', num_params)

## 学習
学習データ（trainval）を用いてU-Net++を学習します．VOC2007のセグメンテーション用データは422枚と少ないため，`epoch_num`を多めに設定しています．

In [ ]:
epoch_num = 30

model.train()
for epoch in range(epoch_num):
    sum_loss = 0.0
    count = 0
    for image, mask in train_loader:
        image, mask = image.to(device), mask.to(device)

        optimizer.zero_grad()
        output = model(image)
        loss = criterion(output, mask)
        loss.backward()
        optimizer.step()

        sum_loss += loss.item() * image.size(0)
        count += image.size(0)

    print(f'epoch: {epoch + 1}, mean loss: {sum_loss / count:.4f}')

## テスト（mIoU評価）
評価用データ（test）に対して，クラスごとのIoU（Intersection over Union）の平均であるmIoU（mean IoU）を求めます．mIoUの算出には，`torchmetrics.classification.MulticlassJaccardIndex`を使用し，無視領域（ラベル255）は`ignore_index`で評価から除外します．

In [ ]:
metric = MulticlassJaccardIndex(num_classes=n_class, ignore_index=IGNORE_INDEX, average='macro').to(device)

model.eval()
with torch.no_grad():
    for image, mask in test_loader:
        image, mask = image.to(device), mask.to(device)
        output = model(image)
        pred = output.argmax(dim=1)
        metric.update(pred, mask)

print(f'mIoU: {metric.compute().item():.4f}')

## セグメンテーション結果の可視化
評価用データから1枚取り出し，入力画像・正解マスク・予測マスクを並べて表示します．

In [ ]:
model.eval()
image, mask = test_data[0]
with torch.no_grad():
    output = model(image.unsqueeze(0).to(device))
    pred = output.argmax(dim=1).squeeze(0).cpu()

# 表示用に正規化前の画素値へ戻す
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
image_vis = (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image_vis); axes[0].set_title('input'); axes[0].axis('off')
axes[1].imshow(decode_segmap(mask)); axes[1].set_title('ground truth'); axes[1].axis('off')
axes[2].imshow(decode_segmap(pred)); axes[2].set_title('prediction'); axes[2].axis('off')
plt.show()

## オリジナルのU-Net++との違い

| 項目 | オリジナル論文 (Zhou et al., 2018) | 本ノートブック |
| :-- | :-- | :-- |
| 学習方法 | Deep Supervision（$X^{0,1}, X^{0,2}, X^{0,3}, X^{0,4}$ すべての出力に損失をかけて学習） | 最終列 $X^{0,4}$ の出力のみに損失をかけて学習（実装簡略化のため） |
| Pruning（推論高速化） | 学習後にDeep Supervisionの中間出力を使って浅いネットワークとして推論を打ち切ることが可能 | 未対応（$X^{0,4}$ の出力のみで推論） |
| バックボーン | スクラッチ学習が基本（VGG/ResNetベースの派生実装も存在） | スクラッチ学習（VOC2007のtrainvalのみ） |
| データ拡張 | マルチスケール学習など複数の工夫を実施 | 固定サイズへのリサイズのみ（データ拡張なし） |
| 学習データ量 | データセットにより異なる（医用画像や大規模データセットで評価） | VOC2007のtrainvalのみ（422枚） |

## 課題

1. `epoch_num`や学習率を変更し，mIoUがどのように変化するか確認してください．
2. 本ノートブックのU-Net++と`unet.ipynb`のU-Netを比較し，パラメータ数やmIoUにどのような違いが出るか確認してください．
3. Deep Supervisionを実装し（$X^{0,1}, X^{0,2}, X^{0,3}, X^{0,4}$ それぞれに1x1畳み込みで出力層を追加し，4つの出力の損失の平均を最終的な損失とする），最終列のみを使った本ノートブックの学習と比べてmIoUがどのように変化するか確認してください．